# Laboratorio 17: generación de imágenes

En este laboratorio exploraremos la generación de imágenes a partir de texto (
**texto-a-imagen**) usando un pipeline de difusión preentrenada de Sana Sprint.

**Al finalizar, podrás:**

1. Cargar un modelo generativo preentrenado para sintetizar imágenes.
2. Relacionar el prompt y los parámetros de inferencia con la imagen generada.

## 1. Preparación del entorno

Instalaremos Diffusers y las bibliotecas necesarias para cargar el modelo,
ejecutar la generación y visualizar el resultado. La instalación requiere
conexión a Internet; en Colab, selecciona un entorno con GPU si está disponible.

In [ ]:
  %pip install --quiet --upgrade diffusers transformers accelerate mediapy peft

## 2. Carga del modelo

La siguiente celda importa las bibliotecas y carga el modelo Sana Sprint de 0.6B
parámetros. Se elige este tamaño porque el modelo de 1.6B puede superar la
memoria disponible en Google Colab. El código usa precisión `bfloat16` y define
`cuda` como dispositivo de carga.

In [ ]:
import mediapy as media
import random
import sys
import torch

from diffusers import SanaSprintPipeline

# NB: the largest model size ("1.6B") does not fit into memory on Google Colab.
model_size = "0.6B"

MAX_IMAGE_SIZE = 1024
model_id = f"Efficient-Large-Model/Sana_Sprint_{model_size}_{MAX_IMAGE_SIZE}px_diffusers"
dtype = torch.bfloat16
device = "cuda"

pipe = SanaSprintPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
).to(device)

## 3. Generación y visualización de una imagen

En esta celda se define una descripción textual y se ejecuta la generación.
`num_inference_steps` controla el número de pasos, `guidance_scale` cuánto se
guía la generación por el prompt, y `width` y `height` fijan la resolución.
La semilla se genera aleatoriamente y se imprime para identificar el resultado.
La tubería se mueve a CPU antes de inferir; esto puede hacer más lenta la
generación.

In [ ]:
prompt = "a photo of Pikachu fine dining with a view to the Eiffel Tower"
seed = random.randint(0, sys.maxsize)

# Pick a value between 1 and 50:
num_inference_steps = 12

# Pick a value between 1.0 and 15.0:
guidance_scale = 5.0

# Pick a value between 256 and MAX_IMAGE_SIZE:
width = MAX_IMAGE_SIZE

# Pick a value between 256 and MAX_IMAGE_SIZE:
height = MAX_IMAGE_SIZE

pipe = pipe.to("cpu")

images = pipe(
    prompt = prompt,
    num_inference_steps = num_inference_steps,
    intermediate_timesteps = None, # this line is required if num_inference_steps != 2
    guidance_scale = guidance_scale,
    width = width,
    height = height,
    generator = torch.Generator("cpu").manual_seed(seed),
    output_type = "pil",
    ).images

print(f"Prompt:\t{prompt}\nSeed:\t{seed}")
media.show_images(images)
images[0].save("output.jpg")

### Ejercicio propuesto

Cambia el prompt para describir una escena distinta y compara dos resultados.
Para que la comparación sea más controlada, conserva la misma semilla; luego
varía un parámetro a la vez (por ejemplo, los pasos o la escala de guía).

## 4. Otros modelos de generación texto-a-imagen

Sana Sprint es una alternativa dentro de una familia amplia de modelos de
difusión.

- **Stable Diffusion 1.x:** una familia conocida para generación texto-a-imagen.
- **Stable Diffusion XL (SDXL):** otra familia de Stable Diffusion, con una
  configuración y requisitos distintos.
- **Sana / Sana Sprint:** la familia usada en este laboratorio; el ejemplo carga
  específicamente Sana Sprint de 0.6B.

### Concepto: controlar una comparación

La generación combina el prompt con un proceso de muestreo que incluye azar.
La **semilla** permite fijar ese azar; así, al comparar dos prompts, mantenemos
constantes la semilla, los pasos, la escala de guía y la resolución. La comparación
no elimina toda variabilidad entre modelos o versiones, pero ayuda a observar el
efecto del cambio que estamos estudiando.

En el ejemplo siguiente solo cambia el nivel de detalle de la descripción. Se
usan imágenes de 512 × 512 para reducir el costo de esta prueba adicional.

In [ ]:
# Comparamos dos descripciones manteniendo los mismos parámetros de generación.
prompts_comparacion = [
    "a small red bicycle",
    "a small red bicycle leaning against a brick wall on a sunny afternoon, "
    "realistic photography, clear details",
]

imagenes_comparacion = []
for prompt_actual in prompts_comparacion:
    resultado = pipe(
        prompt=prompt_actual,
        num_inference_steps=12,
        intermediate_timesteps=None,
        guidance_scale=5.0,
        width=512,
        height=512,
        generator=torch.Generator("cpu").manual_seed(seed),
        output_type="pil",
    )
    imagenes_comparacion.append(resultado.images[0])

# Se reutiliza la semilla del ejemplo anterior para facilitar la comparación.
media.show_images(imagenes_comparacion)